# Emission Impossible — data inspection

A guided look at **every raw source**, how each one is parsed into a tidy table, and
the **exact columns used to join** CO₂ to GDP. Run it from your project folder (the one
with `transform.py` and the `data/` folder).

The tidy schema every source is bent toward is:

| column | meaning |
|---|---|
| `state` | 2-letter code (AL, CA, …) — the join key |
| `year`  | integer year — the join key |
| a value | `co2`, `gdp`, `coal`, … |

So **the join is always `on=["state", "year"]`.** Everything below exists to show you
that each source really does produce those two keys.

In [14]:
import glob, json
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

import transform   # your pipeline module, same folder as this notebook

def latest(source, pattern):
    """Newest file matching data/<source>/<date>/<pattern> (what the pipeline reads)."""
    return sorted(glob.glob(f"data/{source}/*/{pattern}"))[-1]

## 1 · EIA CO₂ — total emissions  *(the main CO₂ source)*

This is a multi-sheet Excel workbook. **Sheet 0 is a "Contents" cover page with no data** —
reading it by accident is what gave us 0 rows earlier. The real numbers live on the
**"Total CO₂"** sheet.

In [15]:
co2_path = latest("eia_co2", "CO2_total.xlsx")
xls = pd.ExcelFile(co2_path)
print("file  :", co2_path)
print("sheets:", xls.sheet_names)          # <- note sheet 0 = 'Contents'

# Sheet 0 — the Contents cover page. All text, no data. THE TRAP.
pd.read_excel(co2_path, sheet_name=0, header=None).head(6)

file  : data/eia_co2/2026-10-02/CO2_total.xlsx
sheets: ['Contents', 'Total CO2', 'Per capita', 'CO2 per billion Btu', 'CO2 per million dollars']


,0,1,2,3,4
0,U.S. Energy Information Administration,NaN,NaN,NaN,NaN
1,State Energy Data System (SEDS),NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,"Release date: June 26, 2026",NaN,NaN,NaN,NaN
4,"Next update: June 25, 2027",NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN


In [16]:
# The real data sheet: a couple of title rows, then 'State' + the years, then AK / AL / ...
pd.read_excel(co2_path, sheet_name="Total CO2", header=None).head(6)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,...,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65
0,"Total energy CO2 emissions, million metric ton...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,State,1960.000,1961.000,1962.000,1963.000,1964.000,1965.000,1966.000,1967.000,1968.000,1969.000,1970.000,1971.000,1972.000,1973.000,...,2010.000,2011.000,2012.000,2013.000,2014.000,2015.000,2016.000,2017.000,2018.000,2019.000,2020.000,2021.000,2022.000,2023.000,2024.000
3,AK,4.040,4.846,5.327,5.366,5.620,5.720,6.812,7.801,8.170,10.014,11.356,12.643,13.432,12.505,...,37.322,37.294,36.401,34.298,34.163,35.073,33.449,33.778,34.573,34.459,36.236,39.969,42.327,43.365,44.008
4,AL,62.359,61.135,66.266,69.121,71.792,79.415,85.121,86.192,93.710,98.439,102.603,98.400,104.846,109.467,...,132.502,129.459,122.582,120.330,122.536,118.910,113.510,108.168,112.021,106.084,98.168,108.543,109.576,101.009,104.323
5,AR,21.284,21.848,23.345,24.756,26.772,26.362,27.914,28.590,31.158,33.719,36.115,35.026,37.094,40.765,...,66.009,67.339,66.127,68.299,68.736,58.801,61.788,63.851,70.495,64.891,54.541,61.836,63.372,60.456,58.415


In [17]:
# transform.load_co2_total() auto-finds the header row + state column and tidies it.
co2 = transform.load_co2_total()           # defaults to the 'Total CO2' sheet
print("shape :", co2.shape)
print("dtypes:", dict(co2.dtypes.astype(str)))
print("years :", co2.year.min(), "-", co2.year.max(), "| states:", co2.state.nunique())
co2.head()

shape : (3315, 3)
dtypes: {'state': 'str', 'year': 'int64', 'co2': 'float64'}
years : 1960 - 2024 | states: 51


,state,year,co2
0,AK,1960,4.040
1,AL,1960,62.359
2,AR,1960,21.284
3,AZ,1960,15.589
4,CA,1960,197.471


## 2 · BEA — real GDP by state  *(JSON)*

BEA returns JSON, not a table. The actual rows are nested at
`BEAAPI → Results → Data`, one record per **state × year**. Note that BEA also returns
*errors* inside this same JSON (with HTTP 200), which is why the loader checks for data
before trusting it.

In [18]:
gdp_path = latest("bea_gdp", "bea_gdp.json")
payload = json.load(open(gdp_path))
print("top-level keys :", list(payload.keys()))
print("BEAAPI keys    :", list(payload["BEAAPI"].keys()))

data = payload["BEAAPI"]["Results"]["Data"]   # <- the rows live here
print("n records      :", len(data))
print("one record     :", data[0])

top-level keys : ['BEAAPI']
BEAAPI keys    : ['Request', 'Results']
n records      : 1740
one record     : {'Code': 'SAGDP9-1', 'GeoFips': '00000', 'GeoName': 'United States', 'TimePeriod': '1997', 'CL_UNIT': 'Millions of chained 2017 dollars', 'UNIT_MULT': '6', 'DataValue': '12370299.0'}


In [19]:
# transform.load_bea_gdp(): GeoName -> state, TimePeriod -> year, DataValue -> gdp
gdp = transform.load_bea_gdp()
print("shape :", gdp.shape)
print("years :", gdp.year.min(), "-", gdp.year.max(), "| states:", gdp.state.nunique())
gdp.head()

shape : (1479, 3)
years : 1997 - 2025 | states: 51


,state,year,gdp
0,AL,2025,259457.1
1,AL,2024,254420.7
2,AL,2023,247878.2
3,AL,2022,240210.5
4,AL,2021,234200.1


## 3 · The join — how CO₂ meets GDP

Both tidy tables carry the **same two key columns, `state` and `year`**. We inner-join on
them. The row count shrinks because the two sources cover **different year spans** — the
join keeps only the years they share.

In [20]:
print("CO2 columns:", list(co2.columns))
print("GDP columns:", list(gdp.columns))
JOIN_KEYS = ["state", "year"]

print("\nCO2 years:", co2.year.min(), "-", co2.year.max())
print("GDP years:", gdp.year.min(), "-", gdp.year.max())
overlap = sorted(set(co2.year) & set(gdp.year))
print("overlapping years:", overlap[0], "-", overlap[-1], f"({len(overlap)} years)")
print("shared states    :", len(set(co2.state) & set(gdp.state)))

df = co2.merge(gdp, on=JOIN_KEYS, how="inner")
print("\nco2 rows:", len(co2), "| gdp rows:", len(gdp), "| JOINED rows:", len(df))

# carbon intensity = tonnes CO2 per $M of real GDP (CO2 is Mt, GDP is $M)
df["intensity"] = df["co2"] / df["gdp"] * 1_000_000
df.head()

CO2 columns: ['state', 'year', 'co2']
GDP columns: ['state', 'year', 'gdp']

CO2 years: 1960 - 2024
GDP years: 1997 - 2025
overlapping years: 1997 - 2024 (28 years)
shared states    : 51

co2 rows: 3315 | gdp rows: 1479 | JOINED rows: 1428


,state,year,co2,gdp,intensity
0,AK,1997,41.383,41071.0,1007.596601
1,AL,1997,134.208,154700.0,867.537169
2,AR,1997,59.169,87180.3,678.696907
3,AZ,1997,71.728,180293.9,397.839306
4,CA,1997,352.738,1441226.4,244.748500


In [21]:
# transform.build_dataset() does exactly this join + intensity, so it should match:
assert len(transform.build_dataset()) == len(df), "build_dataset drifted from the manual join!"
print("build_dataset() matches the manual join:", len(df), "rows")

build_dataset() matches the manual join: 1428 rows


## 4 · The other sources you've collected  *(not yet joined)*

These are already downloaded and ready to bring into the story (fuel mix, sectors, prices).
Each one has the same two join keys hiding in it.

### 4a · CO₂ by fuel — `CO2_source.xlsx`  (Coal / Natural gas / Petroleum)

In [22]:
src_path = latest("eia_co2", "CO2_source.xlsx")
print("sheets:", pd.ExcelFile(src_path).sheet_names)     # Contents + one sheet per fuel + Total

# Same parser, just point it at a fuel sheet (note value_name so the column is labelled):
coal = transform.load_co2_total(path=src_path, sheet="Coal", value_name="coal")
print("Coal tidy shape:", coal.shape, "| years", coal.year.min(), "-", coal.year.max())
coal.head()

sheets: ['Contents', 'Coal', 'Natural gas', 'Petroleum', 'Total']
Coal tidy shape: (3315, 3) | years 1960 - 2024


,state,year,coal
0,AK,1960,0.682
1,AL,1960,36.849
2,AR,1960,0.034
3,AZ,1960,0.020
4,CA,1960,3.272


### 4b · CO₂ by sector — `CO2_sector.xlsx`  (Electric power, Transportation, …)

In [23]:
sec_path = latest("eia_co2", "CO2_sector.xlsx")
print("sheets:", pd.ExcelFile(sec_path).sheet_names)

elec = transform.load_co2_total(path=sec_path, sheet="Electric power", value_name="co2_electric")
print("Electric-power CO2 tidy shape:", elec.shape)
elec.head()

sheets: ['Contents', 'Residential', 'Commercial', 'Industrial', 'Transportation', 'Electric power', 'Total']
Electric-power CO2 tidy shape: (3315, 3)


,state,year,co2_electric
0,AK,1960,0.130
1,AL,1960,17.184
2,AR,1960,2.623
3,AZ,1960,2.942
4,CA,1960,29.074


### 4c · EIA electricity generation — `eia_generation.json`  ⚠️ **truncated**

This call hit the 5,000-row cap, so it only pulled **year 2025, states A–D**. It needs
pagination (loop `offset` / `length`) before it can be used as a time series — don't join
it yet.

In [24]:
gen = json.load(open(latest("eia_generation", "eia_generation.json")))["response"]
print("rows AVAILABLE at EIA :", gen["total"])
print("rows we DOWNLOADED    :", len(gen["data"]))      # capped by length=5000
print("periods present       :", sorted({d["period"] for d in gen["data"]}))
locs = sorted({d["location"] for d in gen["data"]})
print("locations present     :", len(locs), locs[:12])
print("\n>> fix: page through with offset/length before using this source")
pd.DataFrame(gen["data"]).head()

rows AVAILABLE at EIA : 390268
rows we DOWNLOADED    : 5000
periods present       : ['2025']
locations present     : 20 ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'ENC', 'ESC', 'FL']

>> fix: page through with offset/length before using this source


,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units
0,2025,AK,Alaska,1,Electric Utility,ALL,all fuels,5756.2075,thousand megawatthours
1,2025,AK,Alaska,1,Electric Utility,AOR,all renewables,79.83373,thousand megawatthours
2,2025,AK,Alaska,1,Electric Utility,COL,"coal, excluding waste coal",529.47683,thousand megawatthours
3,2025,AK,Alaska,1,Electric Utility,COW,all coal products,529.47683,thousand megawatthours
4,2025,AK,Alaska,1,Electric Utility,DFO,distillate fuel oil,552.07521,thousand megawatthours


### 4d · EIA energy prices — `eia_prices.json`  (complete, ready to use)

In [25]:
pr = json.load(open(latest("eia_prices", "eia_prices.json")))["response"]["data"]
pdf = pd.DataFrame(pr)
print("rows   :", len(pdf))
print("series :", list(pdf.seriesDescription.unique()))
print("keys   : stateId + period  ->  rename to state + year to join")
print("years  :", pdf.period.min(), "-", pdf.period.max(), "| states:", pdf.stateId.nunique())
pdf.head()

rows   : 4368
series : ['Coal price in the electric power sector', 'Electricity price in the residential sector', 'Electricity average price, all sectors']
keys   : stateId + period  ->  rename to state + year to join
years  : 1997 - 2024 | states: 52


,period,seriesId,seriesDescription,stateId,stateDescription,value,unit
0,2024,CLEID,Coal price in the electric power sector,AK,Alaska,4.43,Dollars per million Btu
1,2024,CLEID,Coal price in the electric power sector,AL,Alabama,2.65,Dollars per million Btu
2,2024,CLEID,Coal price in the electric power sector,AR,Arkansas,2.02,Dollars per million Btu
3,2024,CLEID,Coal price in the electric power sector,AZ,Arizona,3.24,Dollars per million Btu
4,2024,CLEID,Coal price in the electric power sector,CA,California,0,Dollars per million Btu


## 5 · Summary — every source, its join keys, and status

In [26]:
summary = pd.DataFrame([
  {"source":"EIA CO2 total", "file":"CO2_total.xlsx",   "join keys":"state, year",
   "rows":len(co2), "years":f"{co2.year.min()}-{co2.year.max()}", "status":"JOINED"},
  {"source":"BEA real GDP",  "file":"bea_gdp.json",      "join keys":"state, year",
   "rows":len(gdp), "years":f"{gdp.year.min()}-{gdp.year.max()}", "status":"JOINED"},
  {"source":"CO2 by fuel",   "file":"CO2_source.xlsx",   "join keys":"state, year",
   "rows":f"{len(coal)}/fuel", "years":f"{coal.year.min()}-{coal.year.max()}", "status":"ready"},
  {"source":"CO2 by sector", "file":"CO2_sector.xlsx",   "join keys":"state, year",
   "rows":f"{len(elec)}/sector", "years":f"{elec.year.min()}-{elec.year.max()}", "status":"ready"},
  {"source":"EIA prices",    "file":"eia_prices.json",   "join keys":"stateId, period",
   "rows":len(pdf), "years":f"{pdf.period.min()}-{pdf.period.max()}", "status":"ready"},
  {"source":"EIA generation","file":"eia_generation.json","join keys":"location, period",
   "rows":f"{len(gen['data'])}/{gen['total']}", "years":"2025 only", "status":"TRUNCATED — fix pagination"},
])
summary

,source,file,join keys,rows,years,status
0,EIA CO2 total,CO2_total.xlsx,"state, year",3315,1960-2024,JOINED
1,BEA real GDP,bea_gdp.json,"state, year",1479,1997-2025,JOINED
2,CO2 by fuel,CO2_source.xlsx,"state, year",3315/fuel,1960-2024,ready
3,CO2 by sector,CO2_sector.xlsx,"state, year",3315/sector,1960-2024,ready
4,EIA prices,eia_prices.json,"stateId, period",4368,1997-2024,ready
5,EIA generation,eia_generation.json,"location, period",5000/390268,2025 only,TRUNCATED — fix pagination
